# Step 2b: Mapping Extracted Items to ESCO Taxonomy


Map each extracted skill/responsibility/requirement to the ESCO taxonomy with confidence scores.

## Why ESCO?
ESCO was chosen as the controlled skills taxonomy because:
- It provides a rich collection of ~13,800 skills and occupations
- It has strong coverage of digital and technical skills
- It is openly licensed and maintained with regular version updates

## Approach
We use a hybrid matching strategy:
1. **Exact matching** on ESCO preferredLabel (fast, precise)
2. **Fuzzy matching** on ESCO altLabels (handles synonyms)
3. **Semantic similarity** using sentence embeddings (handles paraphrasing)

For each extracted item, we return top-3 ESCO matches with confidence scores (0-1)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz, process
import re
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

tqdm.pandas()

## Load Data

In [ ]:
# Load extracted items
extracted_df = pd.read_csv('extracted_items_zero_shot_gpt.csv')
print(f"Extracted items: {len(extracted_df)}")
print(f"Unique ads: {extracted_df['ad_id'].nunique()}")
print(f"\nItem type distribution:")
print(extracted_df['item_type'].value_counts())

extracted_df.head()

In [ ]:
# Load ESCO taxonomy
esco_df = pd.read_csv('skills_en.csv')
print(f"\nESCO skills: {len(esco_df)}")

# Keep relevant columns
esco_df = esco_df[['preferredLabel', 'altLabels', 'description', 'skillType', 'reuseLevel']].copy()
esco_df.head()

## Preprocessing

In [ ]:
def clean_text(text):
    """Clean and normalize text"""
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

# Clean extracted items
extracted_df['extracted_text_clean'] = extracted_df['extracted_text'].apply(clean_text)

# Clean ESCO labels
esco_df['preferredLabel_clean'] = esco_df['preferredLabel'].apply(clean_text)
esco_df['description_clean'] = esco_df['description'].apply(clean_text)

# Parse altLabels (they're newline-separated)
def parse_alt_labels(alt_labels):
    if pd.isna(alt_labels):
        return []
    return [clean_text(label) for label in str(alt_labels).split('\n') if label.strip()]

esco_df['altLabels_list'] = esco_df['altLabels'].apply(parse_alt_labels)

In [ ]:
# Create search index: all possible labels for each ESCO skill
esco_search_index = []

for idx, row in esco_df.iterrows():
    # Add preferred label
    esco_search_index.append({
        'esco_label': row['preferredLabel'],
        'search_term': row['preferredLabel_clean'],
        'match_type': 'preferred'
    })
    
    # Add alt labels
    for alt_label in row['altLabels_list']:
        esco_search_index.append({
            'esco_label': row['preferredLabel'],
            'search_term': alt_label,
            'match_type': 'alt'
        })

esco_search_df = pd.DataFrame(esco_search_index)
print(f"\nESCO search index size: {len(esco_search_df)} (includes alt labels)")

## Method 1: Exact + Fuzzy Matching

In [ ]:
def fuzzy_match_esco(extracted_text, top_k=3, threshold=80):
    """
    Find top-k ESCO matches using fuzzy string matching.
    Returns list of (esco_label, score, match_type)
    """
    if not extracted_text or len(extracted_text.strip()) < 2:
        return []
    
    # Use rapidfuzz for fuzzy matching
    matches = process.extract(
        extracted_text,
        esco_search_df['search_term'].tolist(),
        scorer=fuzz.token_sort_ratio,
        limit=top_k * 3  # Get more candidates, filter later
    )
    
    results = []
    seen_labels = set()
    
    for match_text, score, idx in matches:
        if score < threshold:
            continue
            
        esco_label = esco_search_df.iloc[idx]['esco_label']
        match_type = esco_search_df.iloc[idx]['match_type']
        
        # Avoid duplicates (same ESCO label matched via different alt labels)
        if esco_label not in seen_labels:
            seen_labels.add(esco_label)
            # Normalize score to 0-1
            confidence = score / 100.0
            results.append((esco_label, confidence, match_type))
            
            if len(results) >= top_k:
                break
    
    return results

# Test on a few examples
test_items = extracted_df['extracted_text_clean'].head(5).tolist()
print("Fuzzy matching examples:\n")
for item in test_items:
    matches = fuzzy_match_esco(item, top_k=3)
    print(f"'{item}' →")
    for label, conf, match_type in matches:
        print(f"  - {label} (conf: {conf:.2f}, {match_type})")
    print()

## Method 2: Semantic Similarity

In [ ]:
# Load sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Loaded model: all-MiniLM-L6-v2")

In [ ]:
# Encode ESCO skills (use preferredLabel + description for richer context)
esco_texts = []
for idx, row in esco_df.iterrows():
    text = row['preferredLabel_clean']
    if row['description_clean']:
        text += " " + row['description_clean'][:200]  # Truncate long descriptions
    esco_texts.append(text)

print("Encoding ESCO skills (this may take a few minutes)...")
esco_embeddings = model.encode(esco_texts, show_progress_bar=True, batch_size=64)
print(f"ESCO embeddings shape: {esco_embeddings.shape}")

In [ ]:
def semantic_match_esco(extracted_text, top_k=3):
    """
    Find top-k ESCO matches using semantic similarity.
    Returns list of (esco_label, score)
    """
    if not extracted_text or len(extracted_text.strip()) < 2:
        return []
    
    # Encode extracted text
    query_embedding = model.encode([extracted_text])
    
    # Compute cosine similarity
    similarities = cosine_similarity(query_embedding, esco_embeddings)[0]
    
    # Get top-k indices
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        esco_label = esco_df.iloc[idx]['preferredLabel']
        confidence = float(similarities[idx])
        results.append((esco_label, confidence))
    
    return results

# Test on a few examples
print("\nSemantic matching examples:\n")
for item in test_items:
    matches = semantic_match_esco(item, top_k=3)
    print(f"'{item}' →")
    for label, conf in matches:
        print(f"  - {label} (conf: {conf:.2f})")
    print()

## Hybrid Matching Strategy

In [ ]:
def hybrid_match_esco(extracted_text, top_k=3):
    """
    Hybrid approach: combine fuzzy and semantic matching.
    
    Strategy:
    1. Try fuzzy matching first (fast, good for exact/near-exact matches)
    2. If fuzzy confidence < 0.7, add semantic matches
    3. Merge and re-rank by confidence
    """
    all_matches = {}
    
    # Fuzzy matching
    fuzzy_matches = fuzzy_match_esco(extracted_text, top_k=top_k, threshold=70)
    for label, conf, match_type in fuzzy_matches:
        # Boost confidence if it's a preferred label match
        if match_type == 'preferred':
            conf = min(1.0, conf * 1.05)
        all_matches[label] = max(all_matches.get(label, 0), conf)
    
    # If best fuzzy match is weak, add semantic matches
    best_fuzzy_conf = max([conf for _, conf, _ in fuzzy_matches], default=0)
    if best_fuzzy_conf < 0.7:
        semantic_matches = semantic_match_esco(extracted_text, top_k=top_k)
        for label, conf in semantic_matches:
            # Semantic scores are typically lower, scale them up slightly
            conf_scaled = min(1.0, conf * 1.2)
            all_matches[label] = max(all_matches.get(label, 0), conf_scaled)
    
    # Sort by confidence and return top-k
    sorted_matches = sorted(all_matches.items(), key=lambda x: x[1], reverse=True)
    return sorted_matches[:top_k]

# Test hybrid approach
print("\nHybrid matching examples:\n")
for item in test_items:
    matches = hybrid_match_esco(item, top_k=3)
    print(f"'{item}' →")
    for label, conf in matches:
        print(f"  - {label} (conf: {conf:.2f})")
    print()

## Apply Matching to All Items

In [ ]:
# Apply hybrid matching to all extracted items
print("Matching all extracted items to ESCO...")

results = []

for idx, row in tqdm(extracted_df.iterrows(), total=len(extracted_df)):
    extracted_text = row['extracted_text_clean']
    matches = hybrid_match_esco(extracted_text, top_k=3)
    
    # Store matches
    result = {
        'ad_id': row['ad_id'],
        'ad_title': row['ad_title'],
        'item_type': row['item_type'],
        'extracted_text': row['extracted_text'],
        'context_sentence': row['context_sentence']
    }
    
    for i, (label, conf) in enumerate(matches, 1):
        result[f'esco_match_{i}'] = label
        result[f'confidence_{i}'] = round(conf, 3)
    
    # Fill missing matches with None
    for i in range(len(matches) + 1, 4):
        result[f'esco_match_{i}'] = None
        result[f'confidence_{i}'] = None
    
    results.append(result)

results_df = pd.DataFrame(results)
print(f"\nMatching complete! Total rows: {len(results_df)}")

## Review Results

In [ ]:
# Display sample results
results_df.head(10)

In [ ]:
# Confidence score statistics
conf_cols = ['confidence_1', 'confidence_2', 'confidence_3']
for col in conf_cols:
    print(f"\n{col} statistics:")
    print(results_df[col].describe())

In [ ]:
# Distribution of top confidence scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(conf_cols):
    axes[i].hist(results_df[col].dropna(), bins=20, edgecolor='black', alpha=0.7)
    axes[i].set_xlabel('Confidence Score')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{col.replace("_", " ").title()} Distribution')
    axes[i].axvline(results_df[col].median(), color='red', linestyle='--', label=f'Median: {results_df[col].median():.2f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sample high and low confidence matches
print("\nHigh confidence matches (confidence_1 > 0.9):")
high_conf = results_df[results_df['confidence_1'] > 0.9][['extracted_text', 'esco_match_1', 'confidence_1']].head(10)
for idx, row in high_conf.iterrows():
    print(f"  '{row['extracted_text']}' → '{row['esco_match_1']}' ({row['confidence_1']:.3f})")

print("\nLow confidence matches (confidence_1 < 0.5):")
low_conf = results_df[results_df['confidence_1'] < 0.5][['extracted_text', 'esco_match_1', 'confidence_1']].head(10)
for idx, row in low_conf.iterrows():
    print(f"  '{row['extracted_text']}' → '{row['esco_match_1']}' ({row['confidence_1']:.3f})")

## Save Results

In [ ]:
# Save to CSV
output_file = 'extracted_items_with_esco_matches.csv'
# results_df.to_csv(output_file, index=False)

print(f"Results saved to: {output_file}")
print(f"Shape: {results_df.shape}")
print(f"\nColumns: {list(results_df.columns)}")

## Summary

**Matching Strategy:**
- Hybrid approach combining fuzzy string matching and semantic similarity
- Fuzzy matching handles exact/near-exact matches (fast, precise)
- Semantic matching handles paraphrasing and context (slower, captures semamtic relationships)
- Top-3 ESCO matches per extracted item with confidence scores (0-1)